# **Obtención de resultados con los datos de los entrenos**

In [1]:
%load_ext autoreload
%autoreload 2

import time
import sympy
import numpy as np
from math import log, log2

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

from Crypto.Util import number


from tqdm.notebook import tqdm

import random

from scipy.optimize import curve_fit, least_squares

#Importar de función interna
from modules import schnorr_lattice as sl
from modules import qaoa
from modules import utils

from modules.functions import solve_cvp, solve_cvp_with_opt_paramters

from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)

In [3]:
def generate_results_validation_set(minimum, maximum, n_per_length):
    conjunto = []
    bitlengths = []


    for bit_length in range(minimum, maximum):
        for _ in range(n_per_length):
            N = utils.generate_N(bit_length)

            conjunto.append(N)
            bitlengths.append(bit_length)

    
    df = pd.DataFrame({
        'N': conjunto,
        'Bit_length': bitlengths
    })

    return df


In [4]:
def obtain_data(vnews, instance, b_op, N, n, bitlength,
                probs):

    #Calculo el modulo al cuadrado para comparar las distancias obtenidas
    distances = utils.get_distances2(vnews, instance.t)

    #Calculo la distancia del vector aproximado obtenido clasicamente
    res_vector = np.subtract(b_op, instance.t)
    aprox_distance = np.dot(res_vector, res_vector)

    #Inicializo las variables para vector mas cercano encontrado, vectores simulares al aproximado y vectores mas cortos que el aproximado
    best_distance_prob = 0
    best_distance = np.inf

    aprox_sol_prob = 0

    improve_prob = 0

    for i in range(len(vnews)):

        if distances[i] == aprox_distance: 
            aprox_sol_prob += probs[i]

        if distances[i] < aprox_distance:
            improve_prob += probs[i]
        
        if distances[i] < best_distance:
            best_distance_prob = probs[i]
            best_distance = distances[i]
        elif distances[i] == best_distance: 
            best_distance_prob += probs[i]

    
    return pd.DataFrame({
        'Bit_length':[bitlength],
        'N': [N],
        'Lattice_dimension': [n],
        '|b_op - t|^2': [aprox_distance],
        '|vbest - t|^2': [best_distance],
        'P(b_op)': [aprox_sol_prob],
        'P(vbest)': [best_distance_prob],
        'P(|vnew - t|^2 < |b_op - t|^2)': [improve_prob],
        'P(|vnew - t|^2 <= |b_op - t|^2)': [improve_prob + aprox_sol_prob],
        'P(|vnew - t|^2 > |b_op - t|^2)' : [1 - (improve_prob + aprox_sol_prob)]
    })

In [5]:
def get_empty_df():
    return pd.DataFrame({
            'Bit_length':[],
            'N': [],
            'Lattice_dimension': [],
            '|b_op - t|^2': [],
            '|vbest - t|^2': [],
            'P(b_op)': [],
            'P(vbest)': [],
            'P(|vnew - t|^2 < |b_op - t|^2)': [],
            'P(|vnew - t|^2 <= |b_op - t|^2)': [],
            'P(|vnew - t|^2 > |b_op - t|^2)' : []
        })

In [6]:
def obtain_data_from_valList(res_val_set,
                            infile1: str, infile2: str, outfile1: str, outfile2: str,
                            c, l, seed, set_seed = False, q = 10, normalizeHc = True, delta = 0.75,
                            verbose = False, progress_bar = True):
    
    np.random.seed(seed)

    for p in  tqdm(range(1, 11), desc = 'Escalabilidad') if progress_bar else range(1, 11):

        nelder_df = get_empty_df()
        cobyla_df = get_empty_df()

        with open(f"{infile1}{p}.pkl", "rb") as f:
            angles_nelder = pickle.load(f)
        with open(f"{infile2}{p}.pkl", "rb") as f:
            angles_cobyla = pickle.load(f)

        angles_nelder_list = list(angles_nelder.values())
        angles_cobyla_list = list(angles_cobyla.values())

        for N in res_val_set:
            cvp = sl.schnorrCVP(N, c = c, l = l, seed = seed, set_seed = set_seed, verbose = verbose)
            instance = cvp.generate_cvp(q = q, verbose = verbose)
        
            vnews, probs, b_op, _ = solve_cvp_with_opt_paramters(cvp, instance, angles_nelder_list, delta = delta, normalize = normalizeHc, p = p)
            sol = obtain_data(vnews, instance, b_op, N, cvp.n, cvp.m, probs)
            nelder_df = pd.concat([nelder_df, sol], ignore_index= True)

            vnews, probs, b_op, _ = solve_cvp_with_opt_paramters(cvp, instance, angles_cobyla_list, delta = delta, normalize = normalizeHc, p = p)
            sol = obtain_data(vnews, instance, b_op, N, cvp.n, cvp.m, probs)
            cobyla_df = pd.concat([cobyla_df, sol], ignore_index= True)
        
        nelder_df.to_csv(f"{outfile1}{p}.csv", index = False)
        cobyla_df.to_csv(f"{outfile2}{p}.csv", index = False)   

## **Generación de datos**

In [ ]:
minimum = 8
maximum = 128
n_per_length = 10
setLength = (maximum + 1 - minimum)*n_per_length

#Ejecutado solo para generar un conjunto de validacion final para observar como escala los angulos que hemos encontrado
res_val_df1 = generate_results_validation_set(minimum, maximum + 1, n_per_length)

res_val_df1.to_csv(f'results_vals_sets/resultsValidation_set_size{setLength}_{minimum}to{maximum}_{n_per_length}reps.csv', index = False)

In [13]:
minimum = 9
maximum = 128
n_per_length = 20
setLength = (maximum + 1 - minimum)*n_per_length
#Ejecutado solo para generar un conjunto de validacion final para observar como escala los angulos que hemos encontrado
res_val_df1 = generate_results_validation_set(minimum, maximum + 1, n_per_length)

res_val_df1.to_csv(f'results_vals_sets/resultsValidation_set_size{setLength}_{minimum}to{maximum}_{n_per_length}reps.csv', index = False)

## **Entrenamiento 1**

En la siguiente seccion se va a validar cómo de buenos son los ángulos calculados en la sección de entrenamiento tanto para COBYLA  como Nelder-Mead

### Obtener los datos 1.1

In [ ]:
minimum = 8
maximum = 128
n_per_length = 10
setLength = (maximum + 1 - minimum)*n_per_length
res_val_df1 = pd.read_csv(f'results_vals_sets/resultsValidation_set_size{setLength}_{minimum}to{maximum}_{n_per_length}reps.csv')
res_val_set = res_val_df1['N'].to_numpy()

In [9]:
obtain_data_from_valList(res_val_set,
                         "./results/optimal_angles/train10_val242_NelderMead_p",
                         "./results/optimal_angles/train10_val242_COBYLA_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_1_NelderMead_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_1_COBYLA_p",
                         c = 3, l = 1, seed = seed, set_seed = False, q = 10, normalizeHc = True, delta = 0.75, 
                         verbose = False, progress_bar = True)

Escalabilidad:   0%|          | 0/10 [00:00<?, ?it/s]

### Obtener los datos 1.2

In [13]:
minimum = 9
maximum = 128
n_per_length = 20
setLength = (maximum + 1 - minimum)*n_per_length
res_val_df1 = pd.read_csv(f'results_vals_sets/resultsValidation_set_size{setLength}_{minimum}to{maximum}_{n_per_length}reps.csv')
res_val_set = res_val_df1['N'].to_numpy()

In [14]:
obtain_data_from_valList(res_val_set,
                         "./results/optimal_angles/train10_val242_NelderMead_p",
                         "./results/optimal_angles/train10_val242_COBYLA_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_1_NelderMead_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_1_COBYLA_p",
                         c = 3, l = 1, seed = seed, set_seed = False, q = 10, normalizeHc = True, delta = 0.75, 
                         verbose = False, progress_bar = True)

Escalabilidad:   0%|          | 0/10 [00:00<?, ?it/s]

## **Entrenamiento 2**

### Obtener los datos 2.1

In [11]:
minimum = 8
maximum = 128
n_per_length = 10
setLength = (maximum + 1 - minimum)*n_per_length
res_val_df1 = pd.read_csv(f'results_vals_sets/resultsValidation_set_size{setLength}_{minimum}to{maximum}_{n_per_length}reps.csv')
res_val_set = res_val_df1['N'].to_numpy()


In [12]:
obtain_data_from_valList(res_val_set,
                         "./results/optimal_angles/train50_val480_NelderMead_p",
                         "./results/optimal_angles/train50_val480_COBYLA_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_2_NelderMead_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_2_COBYLA_p",
                         c = 4, l = 1, seed = seed, set_seed = False, q = 10, normalizeHc = True, delta = 0.75, 
                         verbose = False, progress_bar = True)

Escalabilidad:   0%|          | 0/10 [00:00<?, ?it/s]

### Obtener los datos 2.2

In [ ]:
minimum = 9
maximum = 128
n_per_length = 20
setLength = (maximum + 1 - minimum)*n_per_length
res_val_df1 = pd.read_csv(f'results_vals_sets/resultsValidation_set_size{setLength}_{minimum}to{maximum}_{n_per_length}reps.csv')
res_val_set = res_val_df1['N'].to_numpy()

In [ ]:
obtain_data_from_valList(res_val_set,
                         "./results/optimal_angles/train50_val480_NelderMead_p",
                         "./results/optimal_angles/train50_val480_COBYLA_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_2_NelderMead_p",
                         f"./results/scalability_results/datas{setLength}_from{minimum}to{maximum}_2_COBYLA_p",
                         c = 4, l = 1, seed = seed, set_seed = False, q = 10, normalizeHc = True, delta = 0.75, 
                         verbose = False, progress_bar = True)